In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

We need to create data of various formats: nominal, ordinal, interval and ratio.  
Topic: **Online Bookstore** (customers, books, orders, order items).

In [ ]:
n_customers = 300

genders = ["Male", "Female", "Other"]
membership_levels = ["Basic", "Silver", "Gold"]
email_domains = ["gmail.com", "outlook.com", "yahoo.com", "icloud.com", "hotmail.com"]

first_names = [
    "Aarav","Saanvi","Arjun","Priya","Rahul","Neha","Vikram","Ananya","Rohit","Isha",
    "Kiran","Meera","Nikhil","Pooja","Aditi","Suresh","Kavya","Manish","Divya","Ravi",
    "Sneha","Vivek","Naina","Deepak","Swati","Gaurav","Riya","Kunal","Alok","Shreya"
]

last_names = [
    "Sharma","Patel","Singh","Kumar","Yadav","Gupta","Mehta","Reddy","Nair","Jain",
    "Iyer","Kapoor","Bose","Chopra","Mishra","Verma","Saxena","Bhat","Das","Ghosh"
]

initials = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")

# Ensure UNIQUE names
full_names = []
used = set()
while len(full_names) < n_customers:
    name = f"{np.random.choice(first_names)} {np.random.choice(initials)}. {np.random.choice(last_names)}"
    if name not in used:
        used.add(name)
        full_names.append(name)

# Generate realistic emails
emails = []
for name in full_names:
    clean = name.lower().replace(". ", "").replace(" ", "")
    domain = np.random.choice(email_domains)
    number = np.random.randint(10, 999)
    emails.append(f"{clean}{number}@{domain}")


def mask_email(email):
    local, domain = email.split("@")
    keep = local[:2] if len(local) >= 2 else local[:1]
    return keep + "****@" + domain

masked_emails = [mask_email(e) for e in emails]


Most real databases have missing data or otherwise undesirable values, "filler" values.  
We will simulate this by:
- Setting **~5% emails to missing (NULL)**  
- Making **some duplicate names** (realistic)

# 5% missing emails
n_missing = int(0.05 * n_customers)
missing_idx = np.random.choice(df_customers.index, n_missing, replace=False)
df_customers.loc[missing_idx, "email"] = None

# Duplicate names (e.g., 3% duplicates)
n_dupe = int(0.03 * n_customers)
dupe_targets = np.random.choice(df_customers.index, n_dupe, replace=False)
df_customers.loc[dupe_targets, "full_name"] = df_customers.loc[dupe_targets, "full_name"].values

df_customers["email"].isna().sum(), df_customers.duplicated(subset=["full_name"]).sum()

In [ ]:
df_customers = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),   # PRIMARY KEY
    "full_name": full_names,
    "email": masked_emails,  # masked emails used here
    "gender": np.random.choice(genders, n_customers, p=[0.49, 0.49, 0.02]),  # Nominal
    "membership_level": np.random.choice(membership_levels, n_customers, p=[0.70, 0.20, 0.10]),  # Ordinal
    "age": pd.Series(np.random.randint(16, 75, n_customers), dtype="Int64"),  # Ratio
    "signup_year": pd.Series(np.random.randint(2016, 2027, n_customers), dtype="Int64")  # Interval
})


### Creating the Customers Table

This cell generates the **Customers table** for the bookstore database.  
The table stores demographic and account information for each customer.

Each column represents a different data type required in the assignment:

- **customer_id** – Unique identifier for each customer. This acts as the **Primary Key** to ensure that every record in the table is unique.
- **full_name** – The customer's full name generated from predefined first names, initials, and last names.
- **email** – Masked email address used to protect customer privacy while still maintaining a realistic email format.
- **gender** – A **nominal categorical variable** representing the customer's gender.
- **membership_level** – An **ordinal categorical variable** representing the customer's membership tier in the bookstore loyalty program (Basic, Silver, Gold).
- **age** – A **ratio numerical variable** representing the customer's age in years.
- **signup_year** – An **interval numerical variable** representing the year the customer joined the bookstore platform.

Random sampling is used to generate realistic distributions for gender and membership levels.  
The data is stored in a **Pandas DataFrame** named `df_customers`, which will later be exported to CSV and used in the SQLite database.

In [ ]:
mask_email_null = np.random.rand(len(df_customers)) < 0.05
df_customers.loc[mask_email_null, "email"] = None

mask_age = np.random.rand(len(df_customers)) < 0.03
df_customers.loc[mask_age, "age"] = pd.NA

mask_signup = np.random.rand(len(df_customers)) < 0.02
df_customers.loc[mask_signup, "signup_year"] = pd.NA

### Introducing Missing Values (Data Realism)

Real-world databases rarely contain perfectly complete information.  
To make the synthetic dataset more realistic, this step deliberately introduces **missing values** into selected columns of the `df_customers` table.

Random masking is applied using NumPy to simulate incomplete records:

- **Email (5%)** – Some customers may not provide an email address or the data may be unavailable.
- **Age (3%)** – Age information may sometimes be missing due to incomplete user profiles.
- **Signup Year (2%)** – In some cases, historical registration data may not be properly recorded.

The masking is done by generating a random boolean mask and replacing the selected values with `None` or `pd.NA`.  
When exported to SQLite, these values will appear as **NULL**, which accurately represents missing data in relational databases.

Including deliberate missing values improves **database realism** and allows for later data preprocessing and cleaning during analysis.

In [ ]:

print("Missing emails:", df_customers["email"].isna().sum())
print("Missing ages:", df_customers["age"].isna().sum())
print("Missing signup_year:", df_customers["signup_year"].isna().sum())
print("Duplicate names:", df_customers.duplicated(subset=["full_name"]).sum())



Missing emails: 18
Missing ages: 11
Missing signup_year: 4
Duplicate names: 0


### Checking Data Quality: Missing and Duplicate Values

This step verifies the realism and quality of the generated **customers dataset** by checking for missing and duplicate values.

The following checks are performed:

- **Missing Emails** – Counts how many customer records do not have an email address.
- **Missing Ages** – Identifies customers where the age value is missing.
- **Missing Signup Year** – Counts records where the signup year was not recorded.
- **Duplicate Names** – Checks if any duplicate customer names were generated.

These checks help confirm that the dataset includes **deliberately introduced missing values** while maintaining **unique customer identities**.  
Including such validation steps is important for understanding data quality before storing the data in the SQLite database or performing further analysis.

In [ ]:
# Set primary key
df_customers = df_customers.set_index("customer_id")
df_customers.head()

,full_name,email,gender,membership_level,age,signup_year
customer_id,,,,,,
1,Vikram T. Mishra,vi****@yahoo.com,Female,Basic,24,2025
2,Kiran H. Mehta,ki****@hotmail.com,Male,Basic,29,2022
3,Gaurav S. Iyer,ga****@outlook.com,Male,Silver,28,2025
4,Kiran X. Kumar,ki****@outlook.com,Male,Gold,72,2023
5,Ananya X. Singh,an****@yahoo.com,Male,Basic,40,2022


### Setting the Primary Key for the Customers Table

In this step, the **customer_id** column is set as the **index of the DataFrame**, which represents the **Primary Key** for the customers table.

A primary key is a unique identifier that ensures each record in a table can be uniquely distinguished from others.  
By setting **customer_id** as the index, we enforce uniqueness for each customer record in the dataset.

This is consistent with relational database design principles, where every table should have a primary key to maintain data integrity and support relationships with other tables. In later tables, such as the **orders table**, this key will be referenced as a **foreign key** to link customer records with their orders.

The `head()` function is used to display the first few rows of the DataFrame to confirm that the primary key has been set correctly.

In [ ]:
df_customers.to_csv("customers.csv")
print("customers.csv saved successfully.")

customers.csv saved successfully.


### Exporting the Customers Table to CSV

After generating and cleaning the **customers dataset**, the table is exported to a **CSV file** using the `to_csv()` function.

Saving the dataset as a CSV file allows it to be easily imported into **SQLite** to create the corresponding database table. This step is important because the assignment requires submitting an SQLite database containing the generated data.

The exported file **customers.csv** will contain all customer records along with their attributes such as name, masked email, gender, membership level, age, and signup year.

A confirmation message is printed to verify that the file has been successfully saved.

In [ ]:
n_books = 200

# Unique Book IDs
book_ids = [f"BK{str(i).zfill(3)}" for i in range(1, n_books + 1)]

# Realistic title components
main_titles = [
    "The Silent Empire", "Whispers of Destiny", "The Forgotten Path",
    "Echoes of the Storm", "Shadows of the Mind", "The Golden Horizon",
    "Beyond the Crimson Sky", "The Hidden Garden", "Legacy of the Flame",
    "The Lost Chronicles", "Secrets of the Ocean", "The Final Journey",
    "Fragments of Eternity", "The Broken Promise", "Winds of Change",
    "The Last Memory", "Rise of the Phoenix", "The Ancient Code",
    "The Invisible Thread", "The Sacred Forest"
]

subtitles = [
    "A Novel", "A Mystery", "A True Story", "A Journey of Hope",
    "The Untold Story", "A Tale of Courage", "Book One",
    "The Awakening", "A Historical Fiction", "A Psychological Thriller"
]

# Generate unique realistic titles
titles = []
used_titles = set()

while len(titles) < n_books:
    title = np.random.choice(main_titles)

    # 50% chance to add subtitle
    if np.random.rand() > 0.5:
        title = f"{title}: {np.random.choice(subtitles)}"

    if title not in used_titles:
        used_titles.add(title)
        titles.append(title)

# Other columns
genres = ["Fiction", "Non-Fiction", "Mystery", "Romance",
          "Sci-Fi", "Fantasy", "Biography", "Self-Help"]

prices = np.round(np.random.lognormal(mean=2.2, sigma=0.4, size=n_books), 2)
prices = np.clip(prices, 3.99, 79.99)

df_books = pd.DataFrame({
    "book_id": book_ids,
    "title": titles,
    "genre": np.random.choice(genres, n_books),
    "publication_year": pd.Series(np.random.randint(1980, 2027, n_books), dtype="Int64"),
    "price": prices
})

df_books.head()

,book_id,title,genre,publication_year,price
0,BK001,The Final Journey: A True Story,Non-Fiction,1990,10.93
1,BK002,Fragments of Eternity,Self-Help,2002,11.84
2,BK003,Winds of Change: A Psychological Thriller,Romance,2013,9.58
3,BK004,The Hidden Garden,Fantasy,2019,8.04
4,BK005,The Broken Promise: A Psychological Thriller,Self-Help,2002,8.60


### Creating the Books Table

This cell generates the **Books table** for the bookstore database. The table stores information about books available in the online bookstore catalogue.

First, **200 unique book identifiers** are created using the format **BK001, BK002, BK003**, etc. These identifiers act as the **Primary Key** for the books table and ensure that each book can be uniquely referenced in the database.

To create realistic book titles, two lists are defined:
- **main_titles** – a collection of base titles.
- **subtitles** – optional subtitle phrases that can be added to some titles.

A loop is used to generate **unique book titles** by randomly selecting a main title and optionally adding a subtitle. A Python set (`used_titles`) is used to prevent duplicate titles from being generated.

Additional attributes are then generated for each book:
- **genre** – a **nominal categorical variable** representing the book category (e.g., Fiction, Mystery, Fantasy).
- **publication_year** – an **interval numeric variable** representing the year the book was published.
- **price** – a **ratio numeric variable** representing the retail price of the book.

Book prices are generated using a **log-normal distribution**, which produces realistic pricing patterns commonly seen in retail environments where most products fall within a moderate price range.

Finally, all attributes are combined into a **Pandas DataFrame (`df_books`)**, and the `head()` function is used to preview the first few rows of the dataset.

In [ ]:
# Ensure correct types
df_books["publication_year"] = df_books["publication_year"].astype("Int64")

# 10% missing publication_year
mask_pub = np.random.rand(len(df_books)) < 0.10
df_books.loc[mask_pub, "publication_year"] = pd.NA

# 7% missing genre
mask_genre = np.random.rand(len(df_books)) < 0.07
df_books.loc[mask_genre, "genre"] = None

# 5% missing price
mask_price = np.random.rand(len(df_books)) < 0.05
df_books.loc[mask_price, "price"] = np.nan

### Introducing Missing Values in the Books Table

Real-world datasets often contain incomplete or missing information. To simulate realistic data conditions, this step deliberately introduces **missing values** into selected columns of the `df_books` table.

First, the **publication_year** column is explicitly converted to the `Int64` data type. This special pandas integer type allows the column to contain **NULL values (pd.NA)**, which represent missing data.

Random masking is then applied to introduce missing values:

- **Publication Year (10%)** – Some books may not have recorded publication dates, especially older or imported titles.
- **Genre (7%)** – Some books may not yet be classified into a specific genre.
- **Price (5%)** – Occasionally pricing information may be unavailable due to system updates or incomplete catalogue entries.

These missing values are created using NumPy random masks and replaced with `pd.NA`, `None`, or `np.nan`. When the dataset is later exported and loaded into **SQLite**, these values will appear as **NULL**, accurately representing missing data in relational databases.

Including deliberate missing values improves the **realism of the dataset** and allows for later **data preprocessing and data quality analysis**.

In [ ]:
print("=== NULL Summary ===")
print(df_books.isna().sum())

print("\nRows containing NULL values:")
print(df_books[df_books.isna().any(axis=1)].head(10))

=== NULL Summary ===
book_id              0
title                0
genre               14
publication_year    22
price                8
dtype: int64

Rows containing NULL values:
   book_id                                         title      genre  \
1    BK002                         Fragments of Eternity  Self-Help   
2    BK003     Winds of Change: A Psychological Thriller    Romance   
10   BK011            Echoes of the Storm: The Awakening       None   
12   BK013            Rise of the Phoenix: The Awakening  Self-Help   
15   BK016                            The Golden Horizon       None   
16   BK017                           The Lost Chronicles    Romance   
23   BK024            The Last Memory: A Tale of Courage  Self-Help   
30   BK031  Beyond the Crimson Sky: A Historical Fiction    Mystery   
33   BK034                 Rise of the Phoenix: Book One     Sci-Fi   
39   BK040             Echoes of the Storm: A True Story       None   

    publication_year  price  
1        

### Checking Missing Values in the Books Table

This step verifies the presence of **missing values (NULL values)** in the `df_books` dataset.

The code performs two main checks:

- **NULL Summary** – Counts the number of missing values in each column of the books table using `isna().sum()`. This helps confirm that missing values were successfully introduced during the data generation process.
  
- **Rows Containing NULL Values** – Displays the first few rows that contain at least one missing value using `df_books.isna().any(axis=1)`. This allows us to visually inspect how missing data appears in the dataset.

Performing this check helps ensure the dataset contains **realistic incomplete records**, which is common in real-world databases. When exported to **SQLite**, these missing values will be stored as **NULL values** in the database.

In [ ]:
df_books = df_books.set_index("book_id")

df_books.head()

,title,genre,publication_year,price
book_id,,,,
BK001,The Final Journey: A True Story,Non-Fiction,1990,10.93
BK002,Fragments of Eternity,Self-Help,2002,NaN
BK003,Winds of Change: A Psychological Thriller,Romance,<NA>,9.58
BK004,The Hidden Garden,Fantasy,2019,8.04
BK005,The Broken Promise: A Psychological Thriller,Self-Help,2002,8.60


### Setting the Primary Key for the Books Table

In this step, the **book_id** column is set as the **index of the DataFrame**, which represents the **Primary Key** for the books table.

A primary key uniquely identifies each record in a table and ensures that no duplicate identifiers exist. In this database, each book is assigned a unique alphanumeric identifier such as **BK001, BK002, BK003**, which allows the book to be referenced in other tables.

Later in the database, the **book_id** will also be used as a **foreign key** in the **order_items table** to link purchased books with customer orders.

The `head()` function is used to display the first few rows of the DataFrame to confirm that the primary key has been correctly applied.

In [ ]:
df_books.to_csv("books.csv")
print("books.csv saved successfully.")

books.csv saved successfully.


### Exporting the Books Table to CSV

After generating and validating the **books dataset**, the table is exported to a **CSV file** using the `to_csv()` function.

Saving the table as **books.csv** allows the dataset to be easily imported into **SQLite**, where it will form the **Books table** in the relational database.

This step is important because the assignment requires submitting a database that contains the generated tables. Exporting the data ensures that the structured dataset can be stored, shared, and loaded into the database system for further querying and analysis.

A confirmation message is printed to indicate that the file has been successfully saved.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

n_orders = 1200  # >= 1000 requirement

delivery_speeds = ["Standard", "Express", "Next-day"]  # Ordinal
payment_methods = ["Card", "PayPal", "BankTransfer"]   # Nominal
discount_choices = [0, 5, 10, 15, 20]

# Create random order dates
order_dates = pd.to_datetime(
    np.random.choice(pd.date_range("2024-01-01", "2026-03-01"), n_orders)
)

df_orders = pd.DataFrame({
    "order_id": np.arange(1, n_orders + 1),  # PRIMARY KEY
    "customer_id": np.random.choice(df_customers.index.values, n_orders, replace=True),  # FOREIGN KEY
    "order_date": order_dates.strftime("%Y-%m-%d"),
    "order_year": pd.Series(order_dates.year, dtype="Int64"),  # Interval (nullable int)
    "delivery_speed": np.random.choice(delivery_speeds, n_orders, p=[0.75, 0.20, 0.05]),
    "payment_method": np.random.choice(payment_methods, n_orders, p=[0.75, 0.20, 0.05]),
    "discount_pct": pd.Series(
        np.random.choice(discount_choices, n_orders, p=[0.60, 0.15, 0.12, 0.08, 0.05]),
        dtype="Int64"
    ),
    "total_amount": 0.0  # will be computed after order_items
})

df_orders.head()

,order_id,customer_id,order_date,order_year,delivery_speed,payment_method,discount_pct,total_amount
0,1,218,2024-04-12,2024,Standard,Card,0,0.0
1,2,141,2025-03-11,2025,Standard,BankTransfer,10,0.0
2,3,104,2024-09-27,2024,Express,Card,5,0.0
3,4,210,2024-04-16,2024,Standard,Card,0,0.0
4,5,111,2024-03-12,2024,Standard,Card,5,0.0


### Creating the Orders Table

This cell generates the **Orders table**, which records purchase transactions made by customers in the bookstore system.

First, a random seed (`np.random.seed(42)`) is set to ensure that the generated data is **reproducible**. This means the same random dataset can be regenerated if the code is run again.

The table contains **1200 order records**, satisfying the assignment requirement that at least one table must contain **more than 1000 rows**.

Several attributes are generated for each order:

- **order_id** – A unique identifier for each order. This serves as the **Primary Key** of the orders table.
- **customer_id** – References the customer who placed the order. This acts as a **Foreign Key**, linking the orders table to the **customers table**.
- **order_date** – The date when the order was placed.
- **order_year** – Extracted from the order date and represents an **interval numeric variable**.
- **delivery_speed** – Shipping option selected by the customer (Standard, Express, Next-day). This represents an **ordinal categorical variable**.
- **payment_method** – Payment option used for the purchase (Card, PayPal, Bank Transfer). This represents a **nominal categorical variable**.
- **discount_pct** – Discount percentage applied to the order.
- **total_amount** – Placeholder column that will later store the final value of the order based on purchased items.

Random values are generated using **NumPy probability distributions** to simulate realistic customer purchasing behaviour.

The resulting dataset is stored in a **Pandas DataFrame called `df_orders`**, and the `head()` function is used to display the first few rows of the generated orders table.

In [ ]:
# 10% missing discount
mask_disc = np.random.rand(len(df_orders)) < 0.10
df_orders.loc[mask_disc, "discount_pct"] = pd.NA

# 3% missing payment_method
mask_pay = np.random.rand(len(df_orders)) < 0.03
df_orders.loc[mask_pay, "payment_method"] = None

# 2% missing delivery_speed
mask_del = np.random.rand(len(df_orders)) < 0.02
df_orders.loc[mask_del, "delivery_speed"] = None

print("=== ORDERS NULL SUMMARY ===")
print(df_orders.isna().sum())

=== ORDERS NULL SUMMARY ===
order_id            0
customer_id         0
order_date          0
order_year          0
delivery_speed     29
payment_method     31
discount_pct      120
total_amount        0
dtype: int64


### Introducing Missing Values in the Orders Table

Real transactional datasets often contain incomplete records due to system errors, missing information, or delayed data entry.  
To simulate these real-world conditions, this step deliberately introduces **missing values** into selected columns of the `df_orders` table.

Random masking is applied using NumPy to replace some values with `NULL` equivalents:

- **Discount Percentage (10%)** – Some orders may not have discount information recorded.
- **Payment Method (3%)** – Occasionally the payment method may not be stored due to processing errors.
- **Delivery Speed (2%)** – Shipping information may sometimes be incomplete in transactional systems.

These missing values are created using boolean masks and replaced with `pd.NA` or `None`. When exported to **SQLite**, these values will appear as **NULL**, which is the standard representation of missing data in relational databases.

Finally, a **NULL summary** is printed to show the number of missing values in each column, confirming that the missing data has been successfully introduced into the dataset.

In [ ]:
invalid_fk = ~df_orders["customer_id"].isin(df_customers.index)
print("Invalid customer_id FK rows:", int(invalid_fk.sum()))

Invalid customer_id FK rows: 0


### Checking for Invalid Foreign Keys

This step verifies that all **customer_id values in the orders table** correctly reference existing customers in the **customers table**.

Since `customer_id` in the `df_orders` table acts as a **Foreign Key**, it must match a valid **Primary Key** in the `df_customers` table. This check ensures **referential integrity** in the database.

The code identifies any rows where the `customer_id` in the orders table does **not exist** in the customers table index. Such rows would represent **invalid foreign key references**.

Finally, the total number of invalid rows is printed. A result of **0 invalid rows** confirms that every order correctly references an existing customer.

In [ ]:
df_orders = df_orders.set_index("order_id")

df_orders.to_csv("orders.csv")
print("orders.csv saved successfully.")
print("Orders rows:", len(df_orders))

orders.csv saved successfully.
Orders rows: 1200


### Setting the Primary Key and Exporting the Orders Table

In this step, the **order_id** column is set as the **index of the DataFrame**, which represents the **Primary Key** for the orders table.  
A primary key uniquely identifies each order and ensures that every transaction record in the table is distinct.

After setting the primary key, the `df_orders` DataFrame is exported to a **CSV file** using the `to_csv()` function.  
The file **orders.csv** will later be imported into **SQLite** to create the Orders table in the database.

A confirmation message is printed to verify that the file has been saved successfully, and the total number of rows is displayed to confirm that the dataset meets the assignment requirement of having **more than 1000 records**.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

order_items_rows = []

for order_id in df_orders.index:
    # each order has 1–5 different books
    k = np.random.randint(1, 6)
    selected_books = np.random.choice(df_books.index.values, size=k, replace=False)

    for book_id in selected_books:
        qty = pd.Series([np.random.randint(1, 4)], dtype="Int64").iloc[0]  # nullable int
        unit_price = float(df_books.loc[book_id, "price"])
        order_items_rows.append([order_id, book_id, qty, unit_price])

df_order_items = pd.DataFrame(
    order_items_rows,
    columns=["order_id", "book_id", "quantity", "unit_price"]
)

# Deliberate NULLs (realistic)
mask_qty = np.random.rand(len(df_order_items)) < 0.03
df_order_items.loc[mask_qty, "quantity"] = pd.NA

mask_price = np.random.rand(len(df_order_items)) < 0.02
df_order_items.loc[mask_price, "unit_price"] = np.nan

print("ORDER_ITEMS NULL summary")
print(df_order_items.isna().sum())

# Set COMPOSITE PRIMARY KEY in pandas
df_order_items = df_order_items.set_index(["order_id", "book_id"])
df_order_items.head()

ORDER_ITEMS NULL summary
order_id        0
book_id         0
quantity      102
unit_price    208
dtype: int64


quantity  unit_price
order_id book_id                      
1        BK096         3.0        5.79
         BK016         2.0         NaN
         BK031         2.0        7.31
         BK160         1.0       13.83
2        BK173         2.0        7.23

### Creating the Order_Items Table

This cell generates the **Order_Items table**, which stores detailed information about the books included in each customer order.  
This table represents the **many-to-many relationship** between orders and books.

For each order in the `df_orders` table, a random number of books (between **1 and 5**) is selected from the `df_books` table. This simulates realistic purchasing behaviour where a customer may buy multiple books in a single order.

For every selected book, the following attributes are recorded:

- **order_id** – Identifier of the order (Foreign Key referencing the Orders table).
- **book_id** – Identifier of the purchased book (Foreign Key referencing the Books table).
- **quantity** – Number of copies purchased for that book.
- **unit_price** – Price of the book at the time of purchase.

The rows are collected into a list and then converted into a **Pandas DataFrame called `df_order_items`**.

To improve database realism, deliberate missing values are introduced:
- **3% missing values in quantity**
- **2% missing values in unit_price**

These values are replaced with `pd.NA` or `np.nan`, which will appear as **NULL values in SQLite**.

Finally, a **composite primary key** is created using both **order_id and book_id**. This ensures that each combination of order and book appears only once in the table, preventing duplicate entries of the same book within the same order.

The `head()` function is used to display the first few rows of the Order_Items table.

In [ ]:
df_order_items.to_csv("order_items.csv")
print("order_items.csv saved.")

order_items.csv saved.


### Exporting the Order_Items Table to CSV

After generating and validating the **Order_Items dataset**, the table is exported to a **CSV file** using the `to_csv()` function.

The file **order_items.csv** contains detailed transactional data linking orders with the books purchased in each order. This table includes the **order_id**, **book_id**, **quantity**, and **unit_price** attributes.

Saving the table as a CSV file allows it to be easily imported into **SQLite**, where it will form the **Order_Items table** in the relational database. This table also maintains the **composite key relationship (order_id, book_id)** used to uniquely identify each item within an order.

A confirmation message is printed to indicate that the file has been successfully saved.

In [ ]:
tmp = df_order_items.reset_index().copy()
tmp["line_total"] = tmp["quantity"].astype("float") * tmp["unit_price"]

totals = tmp.groupby("order_id")["line_total"].sum(min_count=1).fillna(0).round(2)

df_orders["total_amount"] = df_orders.index.to_series().map(totals).fillna(0).round(2)
df_orders.head()

,customer_id,order_date,order_year,delivery_speed,payment_method,discount_pct,total_amount
order_id,,,,,,,
1,218,2024-04-12,2024,Standard,Card,0,45.82
2,141,2025-03-11,2025,Standard,BankTransfer,10,98.55
3,104,2024-09-27,2024,Express,Card,<NA>,41.61
4,210,2024-04-16,2024,Standard,Card,0,75.48
5,111,2024-03-12,2024,Standard,Card,5,104.03


### Calculating Total Order Amounts

This step calculates the **total value of each order** based on the books purchased in the **Order_Items table**.

First, the composite index of `df_order_items` is reset so that **order_id** and **book_id** become regular columns. A temporary DataFrame (`tmp`) is then created to safely perform calculations.

A new column called **line_total** is computed using the formula:

**line_total = quantity × unit_price**

This represents the total price for each individual book item within an order.

Next, the data is grouped by **order_id**, and the line totals are summed to calculate the **total amount spent in each order**. Missing values are handled using `fillna(0)` to ensure that orders with incomplete item data still produce valid totals.

Finally, these calculated totals are mapped back to the **df_orders** table and stored in the **total_amount** column. This column now represents the final value of each customer order.

The `head()` function is used to display the first few rows of the updated orders table with the calculated totals.

In [ ]:
df_orders.to_csv("orders.csv")
print("orders.csv updated with total_amount.")

orders.csv updated with total_amount.


### Updating and Exporting the Orders Table

After calculating the **total_amount** for each order based on the purchased items, the updated **Orders table** is exported again to a CSV file.

The `to_csv()` function saves the updated DataFrame as **orders.csv**, ensuring that the file now contains the correct **total_amount values** for every order.

This step ensures that the final dataset used in the SQLite database includes the complete transactional information, including the total value of each order.

A confirmation message is printed to verify that the updated file has been successfully saved.

In [ ]:
print("Customers Table Columns:")
print(df_customers.columns)

print("\nBooks Table Columns:")
print(df_books.columns)

print("\nOrders Table Columns:")
print(df_orders.columns)

print("\nOrder Items Table Columns:")
print(df_order_items.columns)

Customers Table Columns:
Index(['full_name', 'email', 'gender', 'membership_level', 'age',
       'signup_year'],
      dtype='object')

Books Table Columns:
Index(['title', 'genre', 'publication_year', 'price'], dtype='object')

Orders Table Columns:
Index(['customer_id', 'order_date', 'order_year', 'delivery_speed',
       'payment_method', 'discount_pct', 'total_amount'],
      dtype='object')

Order Items Table Columns:
Index(['quantity', 'unit_price'], dtype='object')


### Displaying the Schema of Each Table

This step prints the **column names for each table** in the database to verify the structure of the generated datasets.

The following tables are checked:

- **Customers Table** – contains customer profile information such as name, email, gender, membership level, age, and signup year.
- **Books Table** – contains information about books available in the bookstore, including title, genre, publication year, and price.
- **Orders Table** – records purchase transactions made by customers, including order details, delivery options, payment methods, and total order value.
- **Order_Items Table** – stores detailed information about the books included in each order, including quantity and unit price.

Printing the column names helps confirm that each table contains the expected attributes and that the database schema has been correctly generated before importing the data into **SQLite**.